# 05 — Comprehensive Analysis

**Goal:** Comprehensive evaluation of FL-GNN for network intrusion detection on CICIoT2023. Includes baselines (FL+CNN, Centralized GCN, FL+MLP), statistical analysis, hyperparameter sensitivity, communication cost, and per-class analysis.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import copy
import pickle
import joblib
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support as prf
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 13, 'axes.titlesize': 14,
    'legend.fontsize': 10, 'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight'
})

BASE = Path('D:/Project/riset/fl-gnn')
DATA = BASE / 'dataset-clean'
RESULTS = BASE / 'results'
RESULTS.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

USE_AMP = False
print(f'AMP: {USE_AMP}')

torch.manual_seed(42)
np.random.seed(42)
print('Seeds set.')


## 1. Load Data

Load sampled train/test, client graphs, centralized graph, test graph, and label encoder.


In [ ]:
print('Loading sampled data...')
train_npz = np.load(DATA / 'sampled_train.npz')
test_npz = np.load(DATA / 'sampled_test.npz')
X_tr, y_tr = train_npz['X'], train_npz['y']
X_te, y_te = test_npz['X'], test_npz['y']
print(f'Train: {X_tr.shape}, Test: {X_te.shape}')

print('Loading label encoder...')
le = joblib.load(DATA / 'label_encoder.pkl')
class_names = le.classes_
print(f'Classes ({len(class_names)}): {list(class_names[:5])}...')

print('Loading client graphs...')
client_graphs = []
for i in range(5):
    cg = np.load(DATA / f'client_{i}_graph.npz')
    X_c = torch.from_numpy(cg['X']).float().to(device)
    y_c = torch.from_numpy(cg['y']).long().to(device)
    ei_c = torch.from_numpy(cg['edge_index']).long().to(device)
    client_graphs.append({'X': X_c, 'y': y_c, 'edge_index': ei_c})
    sz, esz = cg['X'].shape[0], cg['edge_index'].shape[1]
    print(f'  Client {i}: {sz} nodes, {esz} edges')

print('Loading test graph...')
tst = np.load(DATA / 'test_graph.npz')
test_graph = {
    'X': torch.from_numpy(tst['X']).float().to(device),
    'y': torch.from_numpy(tst['y']).long().to(device),
    'edge_index': torch.from_numpy(tst['edge_index']).long().to(device)
}
tst_sz = tst['X'].shape[0]
tst_esz = tst['edge_index'].shape[1]
print(f'Test graph: {tst_sz} nodes, {tst_esz} edges')

print('Loading centralized train graph...')
cen = np.load(DATA / 'centralized_train_graph.npz')
cen_graph = {
    'X': torch.from_numpy(cen['X']).float().to(device),
    'y': torch.from_numpy(cen['y']).long().to(device),
    'edge_index': torch.from_numpy(cen['edge_index']).long().to(device)
}
cen_sz = cen['X'].shape[0]
cen_esz = cen['edge_index'].shape[1]
print(f'Centralized: {cen_sz} nodes, {cen_esz} edges')

in_dim = cen_graph['X'].shape[1]
n_classes = len(class_names)
print(f'Input dim: {in_dim}, Classes: {n_classes}')


## 2. Evaluation & Training Helpers

Define compute_metrics, train_local, evaluate, and run_fl functions.


In [ ]:
def compute_metrics(y_true, preds):
    acc = accuracy_score(y_true, preds)
    p, r, f1, _ = prf(y_true, preds, average='weighted', zero_division=0)
    pm, rm, f1m, _ = prf(y_true, preds, average='macro', zero_division=0)
    return {'accuracy': acc, 'precision_w': p, 'recall_w': r, 'f1_w': f1,
            'precision_m': pm, 'recall_m': rm, 'f1_m': f1m}

def train_local(model, data, lr=1e-3, wd=5e-4, epochs=3):
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for _ in range(epochs):
        opt.zero_grad()
        if 'edge_index' in data:
            out = model(data['X'], data['edge_index'])
        else:
            out = model(data['X'])
        loss = F.cross_entropy(out, data['y'])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        sched.step()
    return loss.item()

def evaluate(model, data):
    model.eval()
    with torch.no_grad():
        if 'edge_index' in data:
            out = model(data['X'], data['edge_index'])
        else:
            out = model(data['X'])
        loss = F.cross_entropy(out, data['y']).item()
        pred = out.argmax(dim=1).cpu().numpy()
    return loss, pred

def run_fl(model_fn, client_graphs, test_graph, num_rounds=10, local_epochs=3, lr=1e-3, wd=5e-4):
    in_dim = client_graphs[0]['X'].shape[1]
    n_classes = len(torch.unique(torch.cat([g['y'] for g in client_graphs])))
    global_model = model_fn(in_dim, n_classes).to(device)

    for r in range(num_rounds):
        weights, ns = [], []
        for cid, cg in enumerate(client_graphs):
            m = copy.deepcopy(global_model)
            train_local(m, cg, lr, wd, local_epochs)
            weights.append({k: v.detach().cpu() for k, v in m.state_dict().items()})
            ns.append(len(cg['y']))
            del m
            torch.cuda.empty_cache()

        total = sum(ns)
        avg = {k: sum(w[k] * (n/total) for w, n in zip(weights, ns)) for k in weights[0]}
        global_model.load_state_dict({k: v.to(device) for k, v in avg.items()})
        torch.cuda.empty_cache()

    _, preds = evaluate(global_model, test_graph)
    return preds

print('All helper functions defined.')


### Model Architectures

Define GCN, GAT, GraphSAGE, CNN1D, and MLP models.


In [ ]:
from torch_geometric.nn import GCNConv, GATConv, SAGEConv

class GCN(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.conv3 = GCNConv(hidden, out_dim)
        self.dropout = dropout
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv2(x, edge_index).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv3(x, edge_index)
        return x

class GAT(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.3):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden // 4, heads=4, concat=True)
        self.conv2 = GATConv(hidden, hidden // 4, heads=4, concat=True)
        self.conv3 = GATConv(hidden, out_dim, heads=1, concat=False)
        self.dropout = dropout
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv2(x, edge_index).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv3(x, edge_index)
        return x

class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.conv3 = SAGEConv(hidden, out_dim)
        self.dropout = dropout
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv2(x, edge_index).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv3(x, edge_index)
        return x

class CNN1D(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, 3, padding=1)
        self.conv2 = nn.Conv1d(64, 128, 3, padding=1)
        self.conv3 = nn.Conv1d(128, 256, 3, padding=1)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(256, out_dim)
        self.dropout = dropout
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv1(x).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv2(x).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.conv3(x).relu()
        x = self.pool(x).squeeze(-1)
        x = self.fc(x)
        return x

class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.3):
        super().__init__()
        self.lin1 = nn.Linear(in_dim, hidden)
        self.lin2 = nn.Linear(hidden, hidden)
        self.lin3 = nn.Linear(hidden, out_dim)
        self.dropout = dropout
    def forward(self, x):
        x = self.lin1(x).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.lin2(x).relu()
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.lin3(x)
        return x

print('All model architectures defined.')


## 3. Baselines

Three baselines: FL+CNN (no graph), Centralized GCN (no FL), and FL+MLP (no graph).


### 3a. FL+CNN Baseline

Run FL with CNN1D (no graph structure). CNN1D does not use edge_index, so we strip it from client data.


In [ ]:
print('Running FL+CNN...')
cnn_client_graphs = [{'X': g['X'], 'y': g['y']} for g in client_graphs]
cnn_test = {'X': test_graph['X'], 'y': test_graph['y']}
cnn_preds = run_fl(CNN1D, cnn_client_graphs, cnn_test)
cnn_metrics = compute_metrics(test_graph['y'].cpu().numpy(), cnn_preds)
print('FL+CNN results:')
for k, v in cnn_metrics.items():
    print(f'  {k}: {v:.4f}')
del cnn_client_graphs
torch.cuda.empty_cache()


### 3b. Centralized GCN (No FL)

Train a GCN on the full training graph for 10 epochs, then evaluate on the test graph.


In [ ]:
print('Training Centralized GCN...')
n_hidden = 256
cen_model = GCN(in_dim, n_hidden, n_classes, dropout=0.3).to(device)
cen_opt = torch.optim.AdamW(cen_model.parameters(), lr=1e-3, weight_decay=5e-4)
cen_sched = torch.optim.lr_scheduler.CosineAnnealingLR(cen_opt, T_max=10)

for epoch in range(10):
    cen_model.train()
    cen_opt.zero_grad()
    out = cen_model(cen_graph['X'], cen_graph['edge_index'])
    loss = F.cross_entropy(out, cen_graph['y'])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(cen_model.parameters(), max_norm=1.0)
    cen_opt.step()
    cen_sched.step()
    if (epoch + 1) % 5 == 0:
        lv = loss.item()
        print(f'  Epoch {epoch+1}/10, Loss: {lv:.4f}')

cen_loss, cen_preds = evaluate(cen_model, test_graph)
cen_metrics = compute_metrics(test_graph['y'].cpu().numpy(), cen_preds)
print('Centralized GCN results:')
for k, v in cen_metrics.items():
    print(f'  {k}: {v:.4f}')
del cen_model
torch.cuda.empty_cache()


### 3c. FL+MLP Ablation

Run FL with MLP (no graph). MLP does not use edge_index, so strip it from client data.


In [ ]:
print('Running FL+MLP...')
mlp_client_graphs = [{'X': g['X'], 'y': g['y']} for g in client_graphs]
mlp_test = {'X': test_graph['X'], 'y': test_graph['y']}
mlp_preds = run_fl(MLP, mlp_client_graphs, mlp_test)
mlp_metrics = compute_metrics(test_graph['y'].cpu().numpy(), mlp_preds)
print('FL+MLP results:')
for k, v in mlp_metrics.items():
    print(f'  {k}: {v:.4f}')
del mlp_client_graphs
torch.cuda.empty_cache()


## 4. Collect FL-GNN Results

Load the three saved FL models (GCN, GAT, GraphSAGE) and evaluate on the test graph.


In [ ]:
print('Loading FL-trained GNN models...')

model_archs = {
    'GCN': GCN,
    'GAT': GAT,
    'GraphSAGE': GraphSAGE
}

fl_gnn_results = {}
for name in model_archs:
    pt_path = RESULTS / f'{name}_global_model.pt'
    if not pt_path.exists():
        print(f'  WARNING: {pt_path} not found, skipping.')
        continue
    print(f'  Loading {name}...')
    model = model_archs[name](in_dim, 256, n_classes, dropout=0.3).to(device)
    model.load_state_dict(torch.load(pt_path, map_location=device))
    _, preds = evaluate(model, test_graph)
    metrics = compute_metrics(test_graph['y'].cpu().numpy(), preds)
    fl_gnn_results[name] = {'metrics': metrics, 'preds': preds}
    del model
    torch.cuda.empty_cache()

print(f'Loaded {len(fl_gnn_results)} FL-GNN models.')
for name, res in fl_gnn_results.items():
    acc = res['metrics']['accuracy']
    f1m = res['metrics']['f1_m']
    print(f'  {name}: Acc={acc:.4f}, F1_m={f1m:.4f}')


## 5. Full Comparison Table & Bar Chart

Create a DataFrame with all methods and a publication-quality bar chart.


In [ ]:
all_results = {}

for name, res in fl_gnn_results.items():
    all_results[f'FL-{name}'] = res['metrics']

all_results['FL-CNN'] = cnn_metrics
all_results['Centralized GCN'] = cen_metrics
all_results['FL-MLP'] = mlp_metrics

df = pd.DataFrame(all_results).T
print('Full Comparison Table:')
print('=' * 80)
display_cols = ['accuracy', 'f1_w', 'f1_m', 'precision_w', 'recall_w']
df_display = df[display_cols].round(4)
df_display.columns = ['Accuracy', 'F1 (w)', 'F1 (m)', 'Prec (w)', 'Rec (w)']
print(df_display.to_string())
print()

df_display.to_csv(RESULTS / 'comparison_table.csv')
print('Saved to results/comparison_table.csv')

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(df))
width = 0.25
ax.bar(x - width, df['accuracy'], width, label='Accuracy', color='#2e86ab')
ax.bar(x, df['f1_w'], width, label='F1 (weighted)', color='#a23b72')
ax.bar(x + width, df['f1_m'], width, label='F1 (macro)', color='#f18f01')
ax.set_xticks(x)
ax.set_xticklabels(all_results.keys(), rotation=30, ha='right')
ax.set_ylabel('Score')
ax.set_title('Performance Comparison Across Methods')
ax.legend(loc='lower right')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / 'comparison_bar.png')
plt.show()
print('Bar chart saved to results/comparison_bar.png')


## 6. Statistical Analysis (Multiple Runs)

Run FL-GCN and FL-CNN 5 times each with different seeds. Report mean ± std.


In [ ]:
seeds = [42, 43, 44, 45, 46]
stat_methods = ['FL-GCN', 'FL-CNN']

stat_results = {m: {'accuracy': [], 'f1_m': [], 'f1_w': []} for m in stat_methods}

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    print(f'Seed {seed}:')

    preds = run_fl(GCN, client_graphs, test_graph)
    m = compute_metrics(test_graph['y'].cpu().numpy(), preds)
    stat_results['FL-GCN']['accuracy'].append(m['accuracy'])
    stat_results['FL-GCN']['f1_m'].append(m['f1_m'])
    stat_results['FL-GCN']['f1_w'].append(m['f1_w'])
    acc_gcn = m['accuracy']
    f1m_gcn = m['f1_m']
    print(f'  FL-GCN: Acc={acc_gcn:.4f}, F1_m={f1m_gcn:.4f}')

    cnn_cg = [{'X': g['X'], 'y': g['y']} for g in client_graphs]
    cnn_tg = {'X': test_graph['X'], 'y': test_graph['y']}
    preds = run_fl(CNN1D, cnn_cg, cnn_tg)
    m2 = compute_metrics(test_graph['y'].cpu().numpy(), preds)
    stat_results['FL-CNN']['accuracy'].append(m2['accuracy'])
    stat_results['FL-CNN']['f1_m'].append(m2['f1_m'])
    stat_results['FL-CNN']['f1_w'].append(m2['f1_w'])
    acc_cnn = m2['accuracy']
    f1m_cnn = m2['f1_m']
    print(f'  FL-CNN: Acc={acc_cnn:.4f}, F1_m={f1m_cnn:.4f}')
    torch.cuda.empty_cache()

print()
print('Statistical Analysis (Mean ± Std over 5 runs):')
print('=' * 60)
stat_df_list = []
for mname in stat_methods:
    row = {}
    for metric in ['accuracy', 'f1_m', 'f1_w']:
        vals = stat_results[mname][metric]
        row[f'{metric}_mean'] = np.mean(vals)
        row[f'{metric}_std'] = np.std(vals)
    stat_df_list.append(pd.Series(row, name=mname))

stat_df = pd.concat(stat_df_list, axis=1).T
print(stat_df.round(4).to_string())
print()
stat_df.round(4).to_csv(RESULTS / 'statistical_analysis.csv')
print('Saved to results/statistical_analysis.csv')

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(stat_methods))
width = 0.25
plot_metrics = ['accuracy', 'f1_m', 'f1_w']
plot_labels = ['Accuracy', 'F1 (macro)', 'F1 (weighted)']
plot_colors = ['#2e86ab', '#f18f01', '#a23b72']

for i, (metric, label, color) in enumerate(zip(plot_metrics, plot_labels, plot_colors)):
    means = [np.mean(stat_results[m][metric]) for m in stat_methods]
    stds = [np.std(stat_results[m][metric]) for m in stat_methods]
    ax.bar(x + (i - 1) * width, means, width, yerr=stds, label=label,
           color=color, capsize=3, error_kw={'linewidth': 1.5})

ax.set_xticks(x)
ax.set_xticklabels(stat_methods)
ax.set_ylabel('Score')
ax.set_title('Multiple Runs (mean ± std, n=5)')
ax.legend(loc='lower right')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / 'statistical_analysis_bar.png')
plt.show()
print('Bar chart saved.')


## 7. Hyperparameter Sensitivity

Sweep key hyperparameters: number of neighbors (k), hidden dimension, Dirichlet alpha, and client count. Only FL-GCN is run for each sweep to save time.


### 7a. Effect of k (Neighbors)

Rebuild k-NN graphs for each k value and run FL-GCN with GCN model.


In [ ]:
def build_graph(X_cpu, k, dev):
    n = len(X_cpu)
    k_use = min(k, n - 1)
    nn = NearestNeighbors(n_neighbors=k_use, metric='cosine', algorithm='brute', n_jobs=-1)
    nn.fit(X_cpu)
    indices = nn.kneighbors(X_cpu, return_distance=False)
    row = np.repeat(np.arange(n), k_use)
    col = indices.ravel()
    edge_index = np.stack([np.concatenate([row, col]), np.concatenate([col, row])], axis=0).astype(np.int64)
    return torch.from_numpy(edge_index).to(dev)

k_values = [5, 10, 15, 20, 30]
k_results = {'k': [], 'accuracy': [], 'f1_m': []}

print('Sweeping k (neighbors)...')
for k in k_values:
    print(f'  k={k}...')
    rebuilt_clients = []
    for i in range(5):
        npz = np.load(DATA / f'client_{i}_graph.npz')
        Xc = npz['X']
        yc = npz['y']
        ei = build_graph(Xc, k, device)
        rebuilt_clients.append({
            'X': torch.from_numpy(Xc).float().to(device),
            'y': torch.from_numpy(yc).long().to(device),
            'edge_index': ei
        })
    preds = run_fl(GCN, rebuilt_clients, test_graph)
    m = compute_metrics(test_graph['y'].cpu().numpy(), preds)
    k_results['k'].append(k)
    acc_k = m['accuracy']
    f1_k = m['f1_m']
    k_results['accuracy'].append(acc_k)
    k_results['f1_m'].append(f1_k)
    print(f'    Acc={acc_k:.4f}, F1_m={f1_k:.4f}')
    del rebuilt_clients
    torch.cuda.empty_cache()

fig, ax1 = plt.subplots(figsize=(7, 5))
ax1.plot(k_results['k'], k_results['accuracy'], 'o-', label='Accuracy', color='#2e86ab', linewidth=2)
ax1.plot(k_results['k'], k_results['f1_m'], 's--', label='F1 (macro)', color='#f18f01', linewidth=2)
ax1.set_xlabel('k (number of neighbors)')
ax1.set_ylabel('Score')
ax1.set_title('Effect of k on FL-GCN Performance')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_xticks(k_values)
plt.tight_layout()
plt.savefig(RESULTS / 'hyperparam_k.png')
plt.show()
print('Saved to results/hyperparam_k.png')


### 7b. Effect of Hidden Dimension

Run FL-GCN with varying hidden dimensions while keeping all other hyperparameters fixed.


In [ ]:
hidden_values = [64, 128, 256, 512]
hd_results = {'hidden': [], 'accuracy': [], 'f1_m': []}

print('Sweeping hidden dimension...')
for h in hidden_values:
    print(f'  hidden={h}...')
    def make_gcn(d, c):
        return GCN(d, h, c, dropout=0.3)
    preds = run_fl(make_gcn, client_graphs, test_graph)
    m = compute_metrics(test_graph['y'].cpu().numpy(), preds)
    hd_results['hidden'].append(h)
    acc_h = m['accuracy']
    f1_h = m['f1_m']
    hd_results['accuracy'].append(acc_h)
    hd_results['f1_m'].append(f1_h)
    print(f'    Acc={acc_h:.4f}, F1_m={f1_h:.4f}')
    torch.cuda.empty_cache()

fig, ax1 = plt.subplots(figsize=(7, 5))
ax1.plot(hd_results['hidden'], hd_results['accuracy'], 'o-', label='Accuracy', color='#2e86ab', linewidth=2)
ax1.plot(hd_results['hidden'], hd_results['f1_m'], 's--', label='F1 (macro)', color='#f18f01', linewidth=2)
ax1.set_xlabel('Hidden Dimension')
ax1.set_ylabel('Score')
ax1.set_title('Effect of Hidden Dimension on FL-GCN Performance')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_xticks(hidden_values)
plt.tight_layout()
plt.savefig(RESULTS / 'hyperparam_hidden.png')
plt.show()
print('Saved to results/hyperparam_hidden.png')


### 7c. Effect of Dirichlet Alpha

Re-split training data across 5 clients using different Dirichlet alpha values to control non-IID degree. Rebuild client graphs and run FL-GCN.


In [ ]:
def dirichlet_split(X_all, y_all, n_clients=5, alpha=1.0, random_state=42):
    n_classes = len(np.unique(y_all))
    client_indices = [[] for _ in range(n_clients)]
    rng = np.random.default_rng(random_state)
    for c in range(n_classes):
        idx = np.where(y_all == c)[0]
        rng.shuffle(idx)
        proportions = rng.dirichlet(np.repeat(alpha, n_clients))
        proportions = np.maximum(proportions, 1e-3)
        proportions /= proportions.sum()
        proportions = (proportions * len(idx)).astype(int)
        proportions[-1] = len(idx) - proportions[:-1].sum()
        start = 0
        for k in range(n_clients):
            client_indices[k].extend(idx[start:start + proportions[k]].tolist())
            start += proportions[k]
    return client_indices

alpha_values = [0.1, 0.5, 1.0, 10.0]
alpha_results = {'alpha': [], 'accuracy': [], 'f1_m': []}

X_all = train_npz['X']
y_all = train_npz['y']

print('Sweeping Dirichlet alpha...')
for alpha in alpha_values:
    print(f'  alpha={alpha}...')
    ci = dirichlet_split(X_all, y_all, n_clients=5, alpha=alpha, random_state=42)
    alpha_clients = []
    for i in range(5):
        Xc = X_all[ci[i]]
        yc = y_all[ci[i]]
        nc = Xc.shape[0]
        k_nn = min(15, nc - 1)
        nn = NearestNeighbors(n_neighbors=k_nn, metric='cosine', algorithm='brute', n_jobs=-1)
        nn.fit(Xc)
        indices = nn.kneighbors(Xc, return_distance=False)
        row = np.repeat(np.arange(nc), k_nn)
        col = indices.ravel()
        ei = np.stack([np.concatenate([row, col]), np.concatenate([col, row])], axis=0).astype(np.int64)
        ei = torch.from_numpy(ei).to(device)
        alpha_clients.append({
            'X': torch.from_numpy(Xc).float().to(device),
            'y': torch.from_numpy(yc).long().to(device),
            'edge_index': ei
        })
    preds = run_fl(GCN, alpha_clients, test_graph)
    m = compute_metrics(test_graph['y'].cpu().numpy(), preds)
    alpha_results['alpha'].append(alpha)
    acc_al = m['accuracy']
    f1_al = m['f1_m']
    alpha_results['accuracy'].append(acc_al)
    alpha_results['f1_m'].append(f1_al)
    print(f'    Acc={acc_al:.4f}, F1_m={f1_al:.4f}')
    del alpha_clients
    torch.cuda.empty_cache()

fig, ax1 = plt.subplots(figsize=(7, 5))
ax1.semilogx(alpha_results['alpha'], alpha_results['accuracy'], 'o-', label='Accuracy', color='#2e86ab', linewidth=2)
ax1.semilogx(alpha_results['alpha'], alpha_results['f1_m'], 's--', label='F1 (macro)', color='#f18f01', linewidth=2)
ax1.set_xlabel('Dirichlet Alpha (log scale)')
ax1.set_ylabel('Score')
ax1.set_title('Effect of Dirichlet Alpha on FL-GCN Performance')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_xticks(alpha_values)
plt.tight_layout()
plt.savefig(RESULTS / 'hyperparam_alpha.png')
plt.show()
print('Saved to results/hyperparam_alpha.png')


### 7d. Effect of Client Count

Re-split training data across different numbers of clients (3, 5, 10) and run FL-GCN.


In [ ]:
n_client_values = [3, 5, 10]
nc_results = {'n_clients': [], 'accuracy': [], 'f1_m': []}

print('Sweeping client count...')
for nc in n_client_values:
    print(f'  n_clients={nc}...')
    ci = dirichlet_split(X_all, y_all, n_clients=nc, alpha=1.0, random_state=42)
    nc_clients = []
    for i in range(nc):
        Xc = X_all[ci[i]]
        yc = y_all[ci[i]]
        ns = Xc.shape[0]
        k_nn = min(15, ns - 1)
        nn = NearestNeighbors(n_neighbors=k_nn, metric='cosine', algorithm='brute', n_jobs=-1)
        nn.fit(Xc)
        indices = nn.kneighbors(Xc, return_distance=False)
        row = np.repeat(np.arange(ns), k_nn)
        col = indices.ravel()
        ei = np.stack([np.concatenate([row, col]), np.concatenate([col, row])], axis=0).astype(np.int64)
        ei = torch.from_numpy(ei).to(device)
        nc_clients.append({
            'X': torch.from_numpy(Xc).float().to(device),
            'y': torch.from_numpy(yc).long().to(device),
            'edge_index': ei
        })
    preds = run_fl(GCN, nc_clients, test_graph)
    m = compute_metrics(test_graph['y'].cpu().numpy(), preds)
    nc_results['n_clients'].append(nc)
    acc_nc = m['accuracy']
    f1_nc = m['f1_m']
    nc_results['accuracy'].append(acc_nc)
    nc_results['f1_m'].append(f1_nc)
    print(f'    Acc={acc_nc:.4f}, F1_m={f1_nc:.4f}')
    del nc_clients
    torch.cuda.empty_cache()

fig, ax1 = plt.subplots(figsize=(7, 5))
ax1.plot(nc_results['n_clients'], nc_results['accuracy'], 'o-', label='Accuracy', color='#2e86ab', linewidth=2)
ax1.plot(nc_results['n_clients'], nc_results['f1_m'], 's--', label='F1 (macro)', color='#f18f01', linewidth=2)
ax1.set_xlabel('Number of Clients')
ax1.set_ylabel('Score')
ax1.set_title('Effect of Client Count on FL-GCN Performance')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_xticks(n_client_values)
plt.tight_layout()
plt.savefig(RESULTS / 'hyperparam_clients.png')
plt.show()
print('Saved to results/hyperparam_clients.png')


## 8. Communication Cost Analysis

Compute the total bytes and megabytes communicated during FL for each method. Model size is calculated as the number of parameters times 4 bytes (float32).


In [ ]:
def model_size_bytes(model_class, in_dim, hidden, out_dim):
    m = model_class(in_dim, hidden, out_dim)
    total_params = sum(p.numel() for p in m.parameters())
    return total_params * 4  # float32 = 4 bytes

methods_comm = {
    'FL-GCN': GCN,
    'FL-GAT': GAT,
    'FL-GraphSAGE': GraphSAGE,
    'FL-CNN': CNN1D,
    'FL-MLP': MLP
}

n_clients_comm = 5
n_rounds_comm = 10

print('Communication Cost (per method):')
print('=' * 70)
print(f'{"Method":<15} {"Params":<12} {"Bytes/round":<15} {"Total MB":<12}')
print('-' * 54)

comm_data = []
for name, cls in methods_comm.items():
    size_b = model_size_bytes(cls, in_dim, 256, n_classes)
    bytes_per_round = size_b * n_clients_comm
    total_mb = (bytes_per_round * n_rounds_comm) / (1024 * 1024)
    param_count = size_b // 4
    comm_data.append({'Method': name, 'Params': f'{param_count:,}', 'MB': total_mb})
    print(f'{name:<15} {param_count:<12,} {bytes_per_round:<15,} {total_mb:<12.2f}')

print('=' * 70)
comm_df = pd.DataFrame(comm_data)
comm_df.to_csv(RESULTS / 'communication_cost.csv', index=False)
print('Saved to results/communication_cost.csv')


## 9. Per-Class Analysis

Compute per-class F1 for each method. Identify the hardest classes for each method and visualize with a heatmap.


In [ ]:
print('Per-Class F1 Analysis:')
print('=' * 80)

all_method_preds = {}
for name, res in fl_gnn_results.items():
    all_method_preds[f'FL-{name}'] = res['preds']
all_method_preds['FL-CNN'] = cnn_preds
all_method_preds['Centralized GCN'] = cen_preds
all_method_preds['FL-MLP'] = mlp_preds

y_true = test_graph['y'].cpu().numpy()

per_class_f1 = {}
for method, preds in all_method_preds.items():
    _, _, f1, _ = prf(y_true, preds, average=None, zero_division=0)
    per_class_f1[method] = f1

for method in all_method_preds:
    f1_vals = per_class_f1[method]
    sorted_idx = np.argsort(f1_vals)
    print(f'{method} - hardest classes (lowest F1):')
    for idx in sorted_idx[:5]:
        cn = class_names[idx] if idx < len(class_names) else f'Class_{idx}'
        print(f'  {cn:30s} F1={f1_vals[idx]:.4f}')

f1_matrix = np.array([per_class_f1[m] for m in all_method_preds])
fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(f1_matrix, xticklabels=class_names, yticklabels=list(all_method_preds.keys()),
            cmap='YlOrRd', annot=False, cbar_kws={'label': 'F1 Score'}, ax=ax)
ax.set_xlabel('Class')
ax.set_ylabel('Method')
ax.set_title('Per-Class F1 Score Across Methods')
ax.set_xticklabels(class_names, rotation=90, fontsize=6)
plt.tight_layout()
plt.savefig(RESULTS / 'per_class_f1_heatmap.png')
plt.show()
print('Saved to results/per_class_f1_heatmap.png')

f1_df = pd.DataFrame(per_class_f1, index=class_names)
f1_df.to_csv(RESULTS / 'per_class_f1.csv')
print('Saved per-class F1 to results/per_class_f1.csv')
